# Modelo RNN de pronóstico oceánico — Caribe, Pacífico Central y Pacífico Sur

Este notebook documenta, paso a paso, el proceso completo (`pipeline_rnn_oleaje.py`):
desde el CSV unificado hasta un modelo LSTM entrenado y evaluado.

**El objetivo no es solo mostrar el código, sino explicar las decisiones tomadas**
en el camino y por qué se tomaron — varias surgieron de revisar el dataset real,
no de la especificación inicial.

Resumen de las decisiones clave que este notebook justifica:

1. Se eliminó `Category` como variable objetivo/feature: resultó ser una
   re-codificación 1 a 1 de `PhaseName` (fase lunar), no una categoría de
   oleaje real — usarla como target de clasificación era fuga de datos.
2. Las direcciones angulares (`wave_direction`, `ocean_current_direction`)
   se codifican como seno/coseno, no como grados crudos, para evitar el
   salto artificial en el límite 359°→0°.
3. El split train/val/test es **por región**, no un corte de fecha global,
   porque Pacífico Sur tiene un historial mucho más corto que las otras
   dos regiones.
4. La métrica de "accuracy" se calcula con una tolerancia distinta según
   el tipo de variable (relativa, absoluta o circular), en vez de un
   único ±10% para todo.


## 1. Imports y configuración

In [4]:
import os
import json

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, OneHotEncoder

plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 50)


### Hiperparámetros y por qué se eligieron

- **`INPUT_HOURS = 72`**: una única ventana de entrada (3 días de historia) que
  alcanza para los tres horizontes de salida, en vez de armar un dataset de
  entrada distinto para cada horizonte.
- **`HORIZONS = [24, 48, 72]`**: el modelo predice los tres horizontes a la vez
  (multi-output), no un modelo por horizonte.
- **`TRAIN_FRAC / VAL_FRAC = 0.70 / 0.15`**: split cronológico (el resto, 0.15,
  es test).
- Tolerancias de la métrica de "accuracy" — se explican más abajo, en la
  sección de evaluación, junto con el porqué de cada una.


In [5]:
DATA_PATH = "/mnt/user-data/uploads/dataset_final_horario.csv"  # en el proyecto real: ../outputs/dataset_unificado/dataset_final_horario.csv
OUT_DIR = "/home/claude/proyecto/outputs/rnn_oleaje_notebook"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_HOURS = 72
HORIZONS = [24, 48, 72]
MAX_HORIZON = max(HORIZONS)

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

COLS_TO_INTERPOLATE = [
    "sea_surface_temperature", "ocean_current_velocity", "ocean_current_direction",
]
CIRCULAR_COLS = ["wave_direction", "ocean_current_direction"]
LINEAR_REG_TARGETS = [
    "wave_height", "sea_surface_temperature", "ocean_current_velocity",
    "Temp_Promedio", "Temp_Minima", "Temp_Maxima", "Precipitacion", "Vel_Viento",
]
REG_TARGET_COLS = LINEAR_REG_TARGETS + [f"{c}_{t}" for c in CIRCULAR_COLS for t in ("sin", "cos")]

TOLERANCE_REL = 0.10
PRECIP_ABS_TOL_MM = 2.0
ANGLE_ABS_TOL_DEG = 15.0

# Para que el notebook se ejecute en un tiempo razonable como demostración,
# EPOCHS aquí es menor que en pipeline_rnn_oleaje.py (80). Para los
# resultados finales del reporte, correr pipeline_rnn_oleaje.py completo.
EPOCHS = 5
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 2. Carga y exploración inicial

Partimos de `dataset_final_horario.csv`, la salida de `build_dataset_unificado.py`
(el script de EDA que unificó IMN, OpenMeteo y las fases lunares por región y hora).


In [ ]:
df = pd.read_csv("../data/processed/dataset_unificado/dataset_final_horario.csv", parse_dates=["datetime"])
df = df.sort_values(["Region", "datetime"]).reset_index(drop=True)
print(df.shape)
df.head()


In [ ]:
df.groupby("Region")["datetime"].agg(["min", "max", "count"])


**Observación clave**: Pacífico Sur tiene un rango de fechas mucho más corto
que Caribe y Pacífico Central (esto se retoma en la sección de split).


## 3. Nulos reales y su imputación

El resumen del EDA anticipaba nulos en `sea_surface_temperature` (~18%) y en
`ocean_current_velocity`/`ocean_current_direction` (~0.2%). Se confirma aquí,
y se ve que están concentrados en Caribe y Pacífico Central — Pacífico Sur no
tiene nulos en estas columnas.

**Decisión**: interpolación lineal en el tiempo, calculada **por región**
(no globalmente, para no mezclar series de regiones distintas) y con
`limit_direction="both"` para cubrir también los nulos que caen al inicio o
al final de la serie de una región.


In [ ]:
print("Nulos por región (%) ANTES de imputar:")
print(df.groupby("Region")[COLS_TO_INTERPOLATE].apply(lambda g: g.isnull().mean() * 100))


In [ ]:
pieces = []
for region, group in df.groupby("Region", sort=False):
    group = group.sort_values("datetime").set_index("datetime")
    group[COLS_TO_INTERPOLATE] = group[COLS_TO_INTERPOLATE].interpolate(
        method="time", limit_direction="both"
    )
    pieces.append(group.reset_index())
df = pd.concat(pieces, ignore_index=True)
df = df.sort_values(["Region", "datetime"]).reset_index(drop=True)

print("Nulos totales DESPUÉS de imputar:", df[COLS_TO_INTERPOLATE].isnull().sum().sum())


## 4. Por qué se eliminó `Category`

Este fue el hallazgo más importante al revisar el CSV real (no se veía en el
resumen de nulos/filas del EDA). La hipótesis inicial era que `Category` era
una categoría de estado de oleaje/mar. La tabla cruzada de abajo muestra que
en realidad es **una re-codificación exacta de `PhaseName`** (la fase lunar):


In [ ]:
ct = pd.crosstab(df["Category"], df["PhaseName"])
ct


In [ ]:
unicos_por_categoria = df.groupby("Category")["PhaseName"].nunique()
print("PhaseName únicos por valor de Category:", unicos_por_categoria.tolist())
print("-> cada Category corresponde a exactamente 1 PhaseName: es un encoding, no una métrica de oleaje.")


**Por qué esto es un problema real y no solo un detalle**: la fase lunar de
cualquier fecha futura se calcula con un calendario astronómico, no depende
de nada que el modelo tenga que aprender del océano o del clima. Si se
predice `Category` como clasificación mientras `PhaseName` está en las
features de entrada, el modelo llega a ~100% de "accuracy" memorizando la
tabla de arriba — no aprende nada sobre el oleaje. Por eso `Category` se
elimina del pipeline por completo (ni feature ni target); `PhaseName` se
mantiene como feature porque, a diferencia de `Category` como *target*,
usarla como *input* conocido de antemano sí es legítimo.


## 5. Codificación circular de direcciones

`wave_direction` y `ocean_current_direction` son ángulos (0°-360°). El
problema: 359° y 2° son casi el mismo ángulo, pero como número crudo la
diferencia es 357 — tanto el entrenamiento (la pérdida penalizaría un error
que en realidad es pequeño) como cualquier métrica de tolerancia quedarían
distorsionados. La gráfica de abajo lo muestra: a la izquierda, un
histograma de grados crudos con "corte" artificial en 0°/360°; a la derecha,
los mismos datos en el círculo seno/coseno, donde ese corte desaparece.


In [ ]:
sample = df["wave_direction"].dropna().sample(3000, random_state=RANDOM_SEED)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(sample, bins=36, color="steelblue")
axes[0].set_title("wave_direction en grados crudos")
axes[0].set_xlabel("grados")

theta = np.deg2rad(sample)
axes[1].scatter(np.cos(theta), np.sin(theta), s=4, alpha=0.3, color="steelblue")
axes[1].set_title("mismos datos, codificados como (sin, cos)")
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()


In [ ]:
for col in CIRCULAR_COLS:
    rad = np.deg2rad(df[col].values)
    df[f"{col}_sin"] = np.sin(rad)
    df[f"{col}_cos"] = np.cos(rad)

df[[f"{c}_{t}" for c in CIRCULAR_COLS for t in ("sin", "cos")]].describe()


## 6. Features de tiempo cíclico y variables categóricas

In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["datetime"].dt.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["datetime"].dt.hour / 24)
doy = df["datetime"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

region_ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(df[["Region"]])
phase_ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(df[["PhaseName"]])

region_df = pd.DataFrame(region_ohe.transform(df[["Region"]]),
                          columns=region_ohe.get_feature_names_out(["Region"]), index=df.index)
phase_df = pd.DataFrame(phase_ohe.transform(df[["PhaseName"]]),
                         columns=phase_ohe.get_feature_names_out(["PhaseName"]), index=df.index)
df = pd.concat([df, region_df, phase_df], axis=1)

feature_cols = (
    LINEAR_REG_TARGETS
    + [f"{c}_{t}" for c in CIRCULAR_COLS for t in ("sin", "cos")]
    + ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]
    + list(region_ohe.get_feature_names_out(["Region"]))
    + list(phase_ohe.get_feature_names_out(["PhaseName"]))
)
print(f"{len(feature_cols)} features de entrada:")
feature_cols


## 7. Ventaneo: de series de tiempo a ejemplos supervisados

Cada ejemplo de entrenamiento es: **72 horas de historia -> valores en
+24h/+48h/+72h**. Antes de armar las ventanas hay que detectar tramos
horarios contiguos por región (una ventana no puede saltar un hueco de
tiempo, aunque en este dataset ya vimos que no quedaron nulos que generen
huecos después de la interpolación).


In [ ]:
def contiguous_segments(sub_df):
    dt = sub_df["datetime"].values
    gap = np.diff(dt) != np.timedelta64(1, "h")
    return np.concatenate(([0], np.cumsum(gap)))


def build_windows(df, feature_cols, cutoffs):
    splits = {"train": [], "val": [], "test": []}
    for region, sub in df.groupby("Region"):
        train_end, val_end = cutoffs[region]
        sub = sub.reset_index(drop=True).copy()
        sub["_seg"] = contiguous_segments(sub)

        for _, seg in sub.groupby("_seg"):
            seg = seg.reset_index(drop=True)
            n = len(seg)
            span = INPUT_HOURS + MAX_HORIZON
            if n < span:
                continue
            feats = seg[feature_cols].values.astype("float32")
            reg_vals = seg[REG_TARGET_COLS].values.astype("float32")
            for i in range(0, n - span + 1):
                last_idx = i + INPUT_HOURS - 1
                window_end_dt = seg["datetime"].iloc[last_idx]
                split = "train" if window_end_dt <= train_end else (
                    "val" if window_end_dt <= val_end else "test"
                )
                X = feats[i:i + INPUT_HOURS]
                y_reg = {h: reg_vals[last_idx + h] for h in HORIZONS}
                splits[split].append((X, y_reg, region, window_end_dt))
    return splits


## 8. Split temporal — por qué por región y no un corte global

Como vimos en la sección 2, Pacífico Sur tiene un historial mucho más corto.
La gráfica siguiente lo hace explícito: si se aplicara un único corte de
fecha (70% del rango total 2022→2026) para las tres regiones, ese corte
caería *después* de que termina el rango completo de Pacífico Sur — la
región quedaría con **cero ventanas en train**, y el modelo combinado nunca
aprendería nada de ella.

Por eso el corte 70/15/15 se calcula **dentro del rango propio de cada
región**, siempre respetando el orden cronológico (sin mezclar pasado y
futuro dentro de una región).


In [ ]:
rangos = df.groupby("Region")["datetime"].agg(["min", "max"])
fig, ax = plt.subplots(figsize=(10, 2.5))
for i, (region, row) in enumerate(rangos.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], linewidth=8, solid_capstyle="butt")
    ax.text(row["min"], i + 0.15, region, fontsize=9)
ax.set_yticks([])
ax.set_title("Rango de fechas disponible por región")
plt.tight_layout()
plt.show()


In [ ]:
def compute_date_cutoffs_per_region(df):
    cutoffs = {}
    for region, sub in df.groupby("Region"):
        dmin, dmax = sub["datetime"].min(), sub["datetime"].max()
        total_h = (dmax - dmin) / pd.Timedelta(hours=1)
        train_end = dmin + pd.Timedelta(hours=total_h * TRAIN_FRAC)
        val_end = dmin + pd.Timedelta(hours=total_h * (TRAIN_FRAC + VAL_FRAC))
        cutoffs[region] = (train_end, val_end)
    return cutoffs

cutoffs = compute_date_cutoffs_per_region(df)
for region, (train_end, val_end) in cutoffs.items():
    print(f"{region:18s} train->val = {train_end} | val->test = {val_end}")


In [ ]:
splits = build_windows(df, feature_cols, cutoffs)
resumen_splits = pd.DataFrame({
    split: pd.Series([w[2] for w in windows]).value_counts()
    for split, windows in splits.items()
}).fillna(0).astype(int)
resumen_splits.loc["TOTAL"] = resumen_splits.sum()
resumen_splits


In [ ]:
resumen_splits.drop("TOTAL").plot(kind="bar", figsize=(8, 4))
plt.title("Ventanas por región y split")
plt.ylabel("# ventanas")
plt.tight_layout()
plt.show()


Con el split por región, **las tres regiones aparecen en train, val y test**
— a diferencia de lo que hubiera pasado con un corte global.


## 9. Escalado

`StandardScaler` para features y para targets, **ajustado solo con train**
(nunca con val/test, para no filtrar información del futuro hacia el
entrenamiento).


In [ ]:
x_scaler = StandardScaler().fit(np.concatenate([w[0] for w in splits["train"]], axis=0))

n_reg = len(REG_TARGET_COLS)
y_scaler = StandardScaler().fit(
    np.array([[w[1][h] for h in HORIZONS] for w in splits["train"]]).reshape(-1, n_reg)
)

data = {}
for split_name, windows in splits.items():
    X = np.stack([w[0] for w in windows])
    shp = X.shape
    X = x_scaler.transform(X.reshape(-1, shp[-1])).reshape(shp).astype("float32")
    y = {}
    for h in HORIZONS:
        y_h = np.stack([w[1][h] for w in windows]).astype("float32")
        y[h] = y_scaler.transform(y_h).astype("float32")
    data[split_name] = (X, y)

joblib.dump(x_scaler, os.path.join(OUT_DIR, "x_scaler.pkl"))
joblib.dump(y_scaler, os.path.join(OUT_DIR, "y_scaler.pkl"))

X_train, y_train = data["train"]
X_val, y_val = data["val"]
X_test, y_test = data["test"]
print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)


## 10. Arquitectura del modelo

`LSTM(64, return_sequences=True) -> Dropout -> LSTM(32) -> Dropout ->
Dense(32) -> 3 cabezas de salida` (una por horizonte, cada una con 12
valores: las 8 variables lineales + los 4 componentes seno/coseno de las
2 variables angulares).

Es un modelo de **regresión multi-output**, no de clasificación — de ahí que
no haya una capa `softmax` ni una sola "accuracy" de clasificación al final.


In [ ]:
def build_model(n_timesteps, n_features, n_targets):
    inputs = keras.Input(shape=(n_timesteps, n_features), name="ventana_entrada")
    x = layers.LSTM(64, return_sequences=True)(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.2)(x)
    tronco = layers.Dense(32, activation="relu")(x)

    outputs = {f"y_reg_{h}h": layers.Dense(n_targets, name=f"y_reg_{h}h")(tronco) for h in HORIZONS}
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={f"y_reg_{h}h": keras.losses.Huber() for h in HORIZONS},
        metrics={f"y_reg_{h}h": ["mae"] for h in HORIZONS},
    )
    return model

model = build_model(X_train.shape[1], X_train.shape[2], len(REG_TARGET_COLS))
model.summary()


**Por qué Huber y no MSE**: `Precipitacion` tiene picos altos (hasta 113mm
frente a una media de 8mm). MSE penaliza esos picos de forma cuadrática y
puede dominar el entrenamiento; Huber es lineal más allá de cierto umbral,
más robusta a esos outliers.


## 11. Entrenamiento

In [ ]:
y_train_dict = {f"y_reg_{h}h": y_train[h] for h in HORIZONS}
y_val_dict = {f"y_reg_{h}h": y_val[h] for h in HORIZONS}

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train_dict,
    validation_data=(X_val, y_val_dict),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2,
)

model.save(os.path.join(OUT_DIR, "modelo_lstm.keras"))


In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(os.path.join(OUT_DIR, "historial_entrenamiento.csv"), index=False)

plt.plot(hist_df["loss"], label="train")
plt.plot(hist_df["val_loss"], label="val")
plt.title("Pérdida (Huber) por epoch")
plt.xlabel("epoch")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Evaluación — tolerancias específicas por tipo de variable

Un ±10% relativo genérico no tiene sentido para todas las variables:

- **Variables angulares** (`wave_direction`, `ocean_current_direction`):
  se reconstruye el ángulo con `atan2(sin, cos)` y se mide la **distancia
  circular** (no la resta cruda), con tolerancia de **±15°**.
- **`Precipitacion`**: puede valer 0, así que un % relativo es inestable
  cerca de cero. Se usa una tolerancia **absoluta de ±2mm**.
- **Resto** (`wave_height`, `sea_surface_temperature`,
  `ocean_current_velocity`, temperaturas, `Vel_Viento`): tolerancia
  **relativa de ±10%**, que sí tiene sentido en variables que no cruzan
  cero.


In [ ]:
def circular_error_deg(true_deg, pred_deg):
    diff = np.abs(true_deg - pred_deg) % 360
    return np.minimum(diff, 360 - diff)


def evaluate(model, X_test, y_test_scaled, y_scaler):
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    rows = []
    col = {name: j for j, name in enumerate(REG_TARGET_COLS)}

    for h in HORIZONS:
        y_true = y_scaler.inverse_transform(y_test_scaled[h])
        y_pred = y_scaler.inverse_transform(preds[f"y_reg_{h}h"])

        for var in LINEAR_REG_TARGETS:
            j = col[var]
            yt, yp = y_true[:, j], y_pred[:, j]
            mae = np.mean(np.abs(yt - yp))
            rmse = np.sqrt(np.mean((yt - yp) ** 2))
            r2 = 1 - np.sum((yt - yp) ** 2) / max(np.sum((yt - yt.mean()) ** 2), 1e-9)
            if var == "Precipitacion":
                acc = np.mean(np.abs(yt - yp) <= PRECIP_ABS_TOL_MM) * 100
                tol_desc = f"±{PRECIP_ABS_TOL_MM}mm"
            else:
                tol = np.maximum(np.abs(yt) * TOLERANCE_REL, 1e-3)
                acc = np.mean(np.abs(yt - yp) <= tol) * 100
                tol_desc = f"±{int(TOLERANCE_REL*100)}%"
            rows.append(dict(variable=var, horizonte=f"+{h}h", tolerancia=tol_desc,
                              MAE=mae, RMSE=rmse, R2=r2, accuracy=acc))

        for cvar in CIRCULAR_COLS:
            sin_j, cos_j = col[f"{cvar}_sin"], col[f"{cvar}_cos"]
            true_deg = (np.degrees(np.arctan2(y_true[:, sin_j], y_true[:, cos_j])) + 360) % 360
            pred_deg = (np.degrees(np.arctan2(y_pred[:, sin_j], y_pred[:, cos_j])) + 360) % 360
            cerr = circular_error_deg(true_deg, pred_deg)
            rows.append(dict(variable=cvar, horizonte=f"+{h}h", tolerancia=f"±{int(ANGLE_ABS_TOL_DEG)}° circular",
                              MAE=cerr.mean(), RMSE=np.sqrt(np.mean(cerr ** 2)), R2=np.nan,
                              accuracy=np.mean(cerr <= ANGLE_ABS_TOL_DEG) * 100))
    return pd.DataFrame(rows)

resultados = evaluate(model, X_test, y_test, y_scaler)
resultados.to_csv(os.path.join(OUT_DIR, "resultados_test.csv"), index=False)
resultados.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}", "accuracy": "{:.1f}%"}) \
    .background_gradient(subset=["accuracy"], cmap="RdYlGn", vmin=0, vmax=100)


In [ ]:
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(11, 5))
pivot = resultados.pivot(index="variable", columns="horizonte", values="accuracy")
pivot.plot(kind="bar", ax=ax)
ax.axhline(90, color="red", linestyle="--", label="meta 90%")
ax.set_ylabel("accuracy (%)")
ax.set_title("Accuracy por variable y horizonte (tolerancia propia de cada variable)")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 12bis. Resultados finales — corrida completa real (no la demo de arriba)

Las secciones anteriores (con `EPOCHS = 5`) fueron solo para validar que el
pipeline completo corre sin errores y para ilustrar la metodología. **Los
resultados de esta sección son los reales**: salen de correr
`pipeline_rnn_oleaje.py` completo (configurado para hasta 80 epochs) y son
los que van al reporte.

- **Epochs corridas: 11** de 80 máximo — `EarlyStopping` cortó el
  entrenamiento ahí porque `val_loss` dejó de mejorar (se ve claro en la
  curva de abajo: el mínimo de `val_loss` está en la epoch 1, y de ahí en
  adelante sube).
- **Mejor `val_loss`: 0.5518** (epoch 1, restaurada por
  `restore_best_weights=True`).


In [ ]:
hist_real = pd.read_csv("/mnt/user-data/uploads/historial_entrenamiento.csv")
hist_real.index.name = "epoch"

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(hist_real["loss"], label="train", marker="o")
axes[0].plot(hist_real["val_loss"], label="val", marker="o")
axes[0].axvline(hist_real["val_loss"].idxmin(), color="red", linestyle="--",
                 label=f"mejor val_loss (epoch {hist_real['val_loss'].idxmin()})")
axes[0].set_title("Pérdida (Huber) real — corrida completa")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(hist_real["learning_rate"], marker="o", color="darkorange")
axes[1].set_title("Learning rate por epoch (ReduceLROnPlateau)")
axes[1].set_xlabel("epoch")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

print(hist_real[["loss", "val_loss", "learning_rate"]])


**Lectura de la curva real**: el `val_loss` tiene su mínimo en la **primera
epoch** (0.5518) y luego sube de forma casi monótona mientras el `loss` de
train sigue bajando — es un patrón clásico de sobreajuste temprano: el
modelo memoriza train rápido pero deja de generalizar a val casi de
inmediato. `ReduceLROnPlateau` va bajando el learning rate al no ver mejora,
pero no alcanza a revertir la tendencia dentro de las 11 epochs antes de que
`EarlyStopping` corte.

Esto es consistente con los resultados de la sección 12bis: el modelo
aporta señal real en varias variables (R² positivo en la mayoría), pero
hay margen de mejora — por ejemplo, bajar el learning rate inicial,
agregar más regularización, o revisar si 64→32 unidades de LSTM son
demasiadas para la cantidad de señal real disponible en 72h de historia.


In [ ]:
# Resultados reales de la corrida completa (pipeline_rnn_oleaje.py,
# 80 epochs config, early stopping en la epoch 11), tomados de
# resumen_entrenamiento.txt

resultados_finales = pd.DataFrame([
    # horizonte +24h
    dict(variable="wave_height", horizonte="+24h", tolerancia="±10%", MAE=0.2253, RMSE=0.3113, R2=0.388, accuracy=41.6),
    dict(variable="sea_surface_temperature", horizonte="+24h", tolerancia="±10%", MAE=0.4453, RMSE=0.5584, R2=0.726, accuracy=100.0),
    dict(variable="ocean_current_velocity", horizonte="+24h", tolerancia="±10%", MAE=0.3987, RMSE=0.5323, R2=0.354, accuracy=27.0),
    dict(variable="Temp_Promedio", horizonte="+24h", tolerancia="±10%", MAE=1.2383, RMSE=1.4271, R2=-0.397, accuracy=93.9),
    dict(variable="Temp_Minima", horizonte="+24h", tolerancia="±10%", MAE=0.9926, RMSE=1.2525, R2=0.190, accuracy=94.1),
    dict(variable="Temp_Maxima", horizonte="+24h", tolerancia="±10%", MAE=1.0476, RMSE=1.4049, R2=0.251, accuracy=96.0),
    dict(variable="Precipitacion", horizonte="+24h", tolerancia="±2.0mm", MAE=7.8318, RMSE=12.9539, R2=-0.236, accuracy=33.0),
    dict(variable="Vel_Viento", horizonte="+24h", tolerancia="±10%", MAE=2.1037, RMSE=3.1783, R2=0.503, accuracy=36.8),
    dict(variable="wave_direction", horizonte="+24h", tolerancia="±15° circular", MAE=12.7496, RMSE=float("nan"), R2=float("nan"), accuracy=73.1),
    dict(variable="ocean_current_direction", horizonte="+24h", tolerancia="±15° circular", MAE=35.0897, RMSE=float("nan"), R2=float("nan"), accuracy=44.2),
    # horizonte +48h
    dict(variable="wave_height", horizonte="+48h", tolerancia="±10%", MAE=0.2706, RMSE=0.3776, R2=0.104, accuracy=38.9),
    dict(variable="sea_surface_temperature", horizonte="+48h", tolerancia="±10%", MAE=0.3957, RMSE=0.4999, R2=0.782, accuracy=100.0),
    dict(variable="ocean_current_velocity", horizonte="+48h", tolerancia="±10%", MAE=0.4097, RMSE=0.5377, R2=0.340, accuracy=26.3),
    dict(variable="Temp_Promedio", horizonte="+48h", tolerancia="±10%", MAE=1.3423, RMSE=1.5722, R2=-0.696, accuracy=91.1),
    dict(variable="Temp_Minima", horizonte="+48h", tolerancia="±10%", MAE=1.0007, RMSE=1.2677, R2=0.174, accuracy=94.0),
    dict(variable="Temp_Maxima", horizonte="+48h", tolerancia="±10%", MAE=1.0925, RMSE=1.4487, R2=0.203, accuracy=95.9),
    dict(variable="Precipitacion", horizonte="+48h", tolerancia="±2.0mm", MAE=7.7583, RMSE=12.6732, R2=-0.172, accuracy=30.1),
    dict(variable="Vel_Viento", horizonte="+48h", tolerancia="±10%", MAE=2.1513, RMSE=3.2906, R2=0.469, accuracy=36.1),
    dict(variable="wave_direction", horizonte="+48h", tolerancia="±15° circular", MAE=14.8469, RMSE=float("nan"), R2=float("nan"), accuracy=64.9),
    dict(variable="ocean_current_direction", horizonte="+48h", tolerancia="±15° circular", MAE=34.8841, RMSE=float("nan"), R2=float("nan"), accuracy=42.3),
    # horizonte +72h
    dict(variable="wave_height", horizonte="+72h", tolerancia="±10%", MAE=0.2836, RMSE=0.3817, R2=0.091, accuracy=33.9),
    dict(variable="sea_surface_temperature", horizonte="+72h", tolerancia="±10%", MAE=0.4657, RMSE=0.5809, R2=0.706, accuracy=100.0),
    dict(variable="ocean_current_velocity", horizonte="+72h", tolerancia="±10%", MAE=0.4360, RMSE=0.5703, R2=0.256, accuracy=26.0),
    dict(variable="Temp_Promedio", horizonte="+72h", tolerancia="±10%", MAE=1.4113, RMSE=1.6286, R2=-0.825, accuracy=91.1),
    dict(variable="Temp_Minima", horizonte="+72h", tolerancia="±10%", MAE=1.0005, RMSE=1.2667, R2=0.177, accuracy=93.0),
    dict(variable="Temp_Maxima", horizonte="+72h", tolerancia="±10%", MAE=1.2480, RMSE=1.6167, R2=0.009, accuracy=95.0),
    dict(variable="Precipitacion", horizonte="+72h", tolerancia="±2.0mm", MAE=7.9400, RMSE=12.8926, R2=-0.216, accuracy=27.1),
    dict(variable="Vel_Viento", horizonte="+72h", tolerancia="±10%", MAE=2.1829, RMSE=3.2768, R2=0.475, accuracy=37.2),
    dict(variable="wave_direction", horizonte="+72h", tolerancia="±15° circular", MAE=13.0447, RMSE=float("nan"), R2=float("nan"), accuracy=69.1),
    dict(variable="ocean_current_direction", horizonte="+72h", tolerancia="±15° circular", MAE=36.2984, RMSE=float("nan"), R2=float("nan"), accuracy=42.4),
])

resultados_finales.to_csv(os.path.join(OUT_DIR, "resultados_finales_reales.csv"), index=False)
resultados_finales.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}", "accuracy": "{:.1f}%"}) \
    .background_gradient(subset=["accuracy"], cmap="RdYlGn", vmin=0, vmax=100)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
pivot_final = resultados_finales.pivot(index="variable", columns="horizonte", values="accuracy")
pivot_final.plot(kind="bar", ax=ax)
ax.axhline(90, color="red", linestyle="--", label="meta 90%")
ax.set_ylabel("accuracy (%)")
ax.set_title("RESULTADOS FINALES (corrida completa, 11 epochs, early stopping)")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


### Un hallazgo importante al revisar estos resultados: `Temp_Promedio` tiene R² negativo

`Temp_Promedio` cumple la meta de 90%+ de accuracy con tolerancia (91-94%)
en los tres horizontes, pero su **R² es negativo** (-0.40 a -0.83). Esto
significa que el modelo predice *peor* que simplemente usar el promedio
histórico como predicción fija — no está aprendiendo nada útil sobre esta
variable en particular, aunque la métrica de tolerancia diga 90%+.

**Por qué pasa esto y por qué no es una contradicción**: `Temp_Promedio`
varía muy poco de una hora a otra (es casi constante en la escala de horas).
Con tan poca variación, incluso una predicción "perezosa" (cercana al
promedio general) cae dentro de ±10% casi siempre — de ahí el accuracy
alto — pero eso no prueba que el modelo esté aprendiendo el patrón, solo
que la variable es fácil de acertar por tolerancia. `Temp_Minima` y
`Temp_Maxima` sí tienen R² positivo (0.01-0.25), así que ahí el modelo
aporta algo real además de la tolerancia amplia.

**Para el reporte**, vale la pena aclarar esto: reportar solo el 90%+ de
`Temp_Promedio` sin mencionar su R² negativo daría una impresión más
optimista de la que corresponde.


## 13. Conclusiones

**Resultados finales reales** (corrida completa de `pipeline_rnn_oleaje.py`,
80 epochs configuradas, `EarlyStopping` cortó en la **epoch 11**, mejor
`val_loss = 0.5518`) — ver sección 12bis arriba para la tabla y gráfico
completos.

**Variables que sí alcanzan la meta de 90%+ accuracy** (con la tolerancia
propia de cada una), en los tres horizontes:
- `sea_surface_temperature`: 100%
- `Temp_Minima`: 93-94%
- `Temp_Maxima`: 95-96%
- `Temp_Promedio`: 91-94% — **pero con R² negativo** (ver aclaración en la
  sección anterior: el accuracy alto no refleja que el modelo aprendió el
  patrón, sino que la variable varía poco hora a hora).

**Variables por debajo de 90%**, consistente con lo anticipado:
- `wave_direction`: 65-73% (la codificación circular ayudó bastante frente
  a usar grados crudos, pero no alcanza el 90%).
- `ocean_current_direction`: 42-44%.
- `wave_height`: 34-42%.
- `ocean_current_velocity`: 26-27%.
- `Vel_Viento`: 36-37%.
- `Precipitacion`: 27-33% (la más difícil, como se esperaba).

**Conclusión honesta para el reporte**: la meta de 90% de accuracy es
alcanzable y se alcanzó para las variables térmicas/lentas
(`sea_surface_temperature`, `Temp_Minima`, `Temp_Maxima`), con la salvedad
de `Temp_Promedio`. Para oleaje, corrientes, viento y precipitación —las
variables con más valor práctico para navegación/pesca/seguridad
costera— el modelo capta señal real (R² positivo en la mayoría) pero no
llega a 90% bajo ninguna tolerancia razonable; son variables intrínsecamente
más ruidosas hora a hora, y un 90% ahí sería una expectativa poco realista
para cualquier modelo, no una falla específica de esta arquitectura.

**Siguiente paso**: la app de Streamlit (`app_resultados_rnn.py`) permite
explorar estos mismos resultados de forma interactiva, filtrando por región
y por variable, y viendo predicciones individuales ventana por ventana.
